# 🎓 Esercizi di Verifica Finale

### 📖 Come svolgere gli esercizi

- Leggi attentamente la descrizione e il diagramma di ogni esercizio prima di iniziare
- Scrivi il tuo codice solo nelle celle marcate con `# TODO`
- Non modificare le celle di setup e quelle di test/verifica
- Esegui le celle di verifica alla fine di ogni esercizio per controllare la tua soluzione

### ✅ Prerequisiti

- Ollama in esecuzione su `localhost:11434` con il modello `llama3.2` scaricato
- Pacchetti già scaricati: `chromadb`, `sentence-transformers`, `ollama`, `langchain`, `langchain-ollama`

```bash
pip install chromadb sentence-transformers ollama langchain langchain-ollama langchain-core
```

---
## ⚙️ Setup globale — esegui questa cella prima di iniziare

In [1]:
%pip install -q chromadb sentence-transformers ollama langchain langchain-ollama langchain-core

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Import globali ────────────────────────────────────────────────────────────
import re
import time
import textwrap
from typing import Any

import chromadb
from chromadb.utils import embedding_functions
import ollama as ollama_client

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Configurazione Ollama ─────────────────────────────────────────────────────
OLLAMA_MODEL = "llama3.2"
BASE_URL     = "http://localhost:11434"

# Verifica connessione
try:
    resp = ollama_client.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": "Rispondi solo: OK"}]
    )
    print(f"✅ Ollama risponde correttamente — modello: {OLLAMA_MODEL}")
except Exception as e:
    print(f"❌ Errore di connessione a Ollama: {e}")
    print("   Assicurati che Ollama sia in esecuzione: ollama serve")

# Helper di stampa
def stampa(titolo: str, testo: str, larghezza: int = 70):
    sep = "─" * 64
    print(f"\n{sep}")
    print(f"📌 {titolo}")
    print(sep)
    print(textwrap.fill(testo.strip(), width=larghezza))
    print()

✅ Ollama risponde correttamente — modello: llama3.2


---
# 📝 Esercizio 1 — Chatbot di supporto per una libreria con RAG
---

## Descrizione

La libreria indipendente **Pagina & Co.** vuole un assistente virtuale che risponda alle domande
dei clienti sul catalogo libri, gli autori, le politiche del negozio e gli eventi in programma.

La knowledge base è già fornita come stringa Python nella cella qui sotto — non devi leggere nessun file.

Devi costruire una **pipeline RAG completa** in 5 step:

```
RAW_TEXT (stringa Python)
        │
        ▼
  [STEP 1] Cleaning del testo [Già svolto]
        │
        ▼
  [STEP 2] Chunking con overlap [Già svolto]
        │
        ▼
  [STEP 3] Indicizzazione su ChromaDB
        │
  Query cliente ──▶ [STEP 4] Retrieval (top-K chunk)
                                   │
                             [STEP 5] Generazione risposta con Ollama
                                   │
                               Risposta 🤖
```

---

## Specifiche tecniche

| Step | Cosa fare | Dettagli |
|------|-----------|----------|
| 3 | Creare una ChromaDB collection | Usa `paraphrase-multilingual-MiniLM-L12-v2`, distanza coseno |
| 4 | Implementare il retrieval | Recupera i **4 chunk** più simili alla query |
| 5 | Generare la risposta | System prompt: assistente Pagina & Co., risponde solo con le info del contesto |

---
### Knowledge base — Pagina & Co.

In [3]:
# ── Knowledge base della libreria Pagina & Co. ────────────────────────────────
# Questa stringa è la tua fonte dati. Non modificarla.

RAW_TEXT = """
GUIDA DELLA LIBRERIA PAGINA & CO.
Via dei Libri 7, 10122 Torino (TO)
Aggiornamento: Settembre 2024
========================================

SEZIONE 1 - PROFILO DELLA LIBRERIA
-----------------------------------

Pagina & Co. è una libreria indipendente fondata nel 2008 a Torino.
Si specializza in narrativa contemporanea, saggistica e letteratura per ragazzi.
Il negozio è aperto dal lunedì al sabato, dalle 9:30 alle 19:30.
La domenica è aperto solo nelle domeniche di novembre e dicembre, dalle 10:00 alle 18:00.
Il personale è composto da 6 librai, tutti con formazione umanistica.
La libreria fa parte del circuito nazionale LibriLiberi, che offre sconti incrociati tra oltre 200 librerie indipendenti.


SEZIONE 2 - CATALOGO E GENERI
-------------------------------

Il catalogo conta circa 8.000 titoli in negozio e accesso a oltre 200.000 titoli ordinabili in 48 ore.
I generi principali disponibili a scaffale sono:
- Narrativa italiana e straniera contemporanea
- Saggistica (storia, filosofia, scienza, economia)
- Letteratura per ragazzi (0-5 anni, 6-10 anni, 11-14 anni, Young Adult)
- Fumetti e graphic novel
- Poesia e teatro
- Libri di cucina e gastronomia
- Guide di viaggio e cartografia

La sezione "Consigli dei librai" espone ogni settimana 10 titoli selezionati dallo staff, con una scheda di lettura scritta a mano.
I libri fuori catalogo o esauriti possono essere cercati tramite il servizio di ricerca antiquaria, attivo su appuntamento.


SEZIONE 3 - PREZZI E SCONTI
-----------------------------

I prezzi rispettano la Legge Levi (legge 128/2011), che limita gli sconti sui libri al 5% per i privati.
Eccezioni previste dalla legge:
- Fiere del libro: sconto fino al 20%
- Vendite a enti (scuole, biblioteche, comuni): sconto fino al 15%
- Remainders (libri con prezzo di copertina barrato): nessun limite di sconto

La tessera fedeltà "Carta Pagina" costa 10 euro all'anno e garantisce:
- Sconto del 5% su tutti gli acquisti (cumulabile con lo sconto legge Levi)
- Accesso prioritario agli eventi e presentazioni
- Newsletter mensile con anteprime e consigli di lettura
- Un segnalibro artigianale in omaggio all'iscrizione


SEZIONE 4 - ORDINI E SPEDIZIONI
---------------------------------

È possibile ordinare qualsiasi titolo disponibile presso i distributori nazionali.
I tempi di consegna standard sono:
- Titoli italiani in catalogo: 1-2 giorni lavorativi
- Titoli stranieri in catalogo: 3-5 giorni lavorativi
- Titoli fuori catalogo o esauriti: tempi variabili, preventivo su richiesta

La spedizione a domicilio è disponibile su tutto il territorio italiano.
Costo di spedizione: 3,90 euro per ordini sotto i 25 euro; gratuita per ordini pari o superiori a 25 euro.
Il ritiro in negozio è sempre gratuito e disponibile entro 2 ore dalla conferma dell'ordine.


SEZIONE 5 - EVENTI E ATTIVITÀ
--------------------------------

Pagina & Co. organizza regolarmente eventi culturali:

5.1 PRESENTAZIONI DI LIBRI
Si tengono in media due volte al mese, solitamente il giovedì sera alle 18:30.
L'ingresso è sempre gratuito e aperto a tutti.
Gli autori presenti nel 2024 includono Donatella Di Pietrantonio, Paolo Cognetti e Viola Ardone.

5.2 CLUB DEL LIBRO
Il club si riunisce ogni primo martedì del mese alle 19:00.
Ogni incontro dura circa 90 minuti e si discute un libro scelto con un mese di anticipo.
L'iscrizione annuale al club costa 20 euro (include 10 incontri, tè e biscotti offerti dalla libreria).
Il prossimo titolo in programma è "Spatriati" di Mario Desiati.

5.3 LABORATORI PER BAMBINI
Ogni sabato mattina dalle 10:00 alle 11:30 si tiene un laboratorio di lettura e creatività
per bambini dai 4 agli 8 anni. Il costo è di 8 euro a bambino per sessione.
I laboratori sono condotti da Marta Ferretti, educatrice e illustratrice.

5.4 ESTATE LETTERARIA
Da giugno ad agosto la libreria organizza l'Estate Letteraria:
letture ad alta voce nel cortile interno ogni venerdì sera alle 21:00.
L'ingresso è gratuito, i posti sono limitati a 40 persone (prenotazione consigliata).


SEZIONE 6 - POLITICHE E SERVIZI
----------------------------------

6.1 RESI E CAMBI
I libri possono essere restituiti entro 14 giorni dall'acquisto se in perfette condizioni
e accompagnati dallo scontrino. Non si accettano resi su libri con dediche o evidenziazioni.
Per i libri acquistati online, il reso è gratuito entro 14 giorni.

6.2 BUONI REGALO
I buoni regalo sono disponibili da 10, 25, 50 e 100 euro.
Non hanno scadenza e possono essere usati anche per ordini online.
Sono acquistabili in negozio o tramite il sito www.paginaeco.it.

6.3 ADOZIONE DI UN LIBRO
Il progetto "Adotta un libro" permette ai clienti di acquistare un libro
da donare alla biblioteca scolastica convenzionata più vicina.
I libri donati ricevono un ex libris con il nome del donatore.

6.4 ACCESSIBILITÀ
La libreria è accessibile alle sedie a rotelle. È disponibile un servizio di ascolto
dedicato alle persone con difficoltà visive: su prenotazione, un libraio legge
la quarta di copertina e i primi capitoli di qualsiasi libro.


SEZIONE 7 - CONTATTI
----------------------

Indirizzo: Via dei Libri 7, 10122 Torino (TO)
Telefono: 011 456 7890
Email: info@paginaeco.it
Sito web: www.paginaeco.it
Instagram: @paginaeco
Orari: lun-sab 9:30-19:30 | dom (solo nov-dic) 10:00-18:00
"""

print(f"✅ Knowledge base caricata: {len(RAW_TEXT)} caratteri")

✅ Knowledge base caricata: 5284 caratteri


---
### STEP 1 — Cleaning del testo

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Cleaning del testo
# ─────────────────────────────────────────────────────────────────────────────

def clean_text(text: str) -> str:
    """
    Pulisce il testo grezzo rimuovendo elementi di rumore.

    Parametri
    ----------
    text : testo grezzo da pulire

    Ritorna
    -------
    testo pulito
    """
    # 1. Rimuove le righe composte solo da "=" o "-" (almeno 3 caratteri)
    text = re.sub(r'^[=\-]{3,}\s*$', '', text, flags=re.MULTILINE)

    # 2. Strip di ogni riga
    lines = [line.strip() for line in text.splitlines()]
    text = "\n".join(lines)

    # 3. Normalizza newline multipli (max 2 consecutivi)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # 4. Rimuove spazi multipli interni alla riga
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()


cleaned_text = clean_text(RAW_TEXT)
print(f"✅ Testo pulito: {len(cleaned_text)} caratteri (originale: {len(RAW_TEXT)})")
print(f"   Riduzione: {(1 - len(cleaned_text)/len(RAW_TEXT))*100:.1f}%")

✅ Testo pulito: 5012 caratteri (originale: 5284)
   Riduzione: 5.1%


---
### STEP 2 — Chunking con overlap

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Chunking con overlap
# ─────────────────────────────────────────────────────────────────────────────

def split_into_chunks(
    text: str,
    chunk_size: int = 500,
    overlap: int = 100,
    separator: str = "\n\n",
    source: str = "pagina_e_co"
) -> list[dict]:
    """
    Divide il testo in chunk con overlap.

    Parametri
    ----------
    text       : testo da dividere
    chunk_size : dimensione massima di ogni chunk (caratteri)
    overlap    : caratteri da sovrapporre tra chunk consecutivi
    separator  : separatore di paragrafi
    source     : nome del file sorgente (per i metadati)

    Ritorna
    -------
    lista di dict con chiavi 'text' e 'metadata'
    """
    paragrafi = [p.strip() for p in text.split(separator) if p.strip()]

    chunks = []
    chunk_corrente = ""
    chunk_index = 0

    for para in paragrafi:
        if len(chunk_corrente) + len(para) > chunk_size and chunk_corrente:
            # Salva il chunk corrente
            chunks.append({
                "text": chunk_corrente.strip(),
                "metadata": {"source": source, "chunk_index": chunk_index}
            })
            # Nuovo chunk: inizia con l'overlap del precedente
            chunk_corrente = chunk_corrente[-overlap:] + " " + para
            chunk_index += 1
        else:
            chunk_corrente = (chunk_corrente + " " + para).strip() if chunk_corrente else para

    # Salva l'ultimo chunk
    if chunk_corrente.strip():
        chunks.append({
            "text": chunk_corrente.strip(),
            "metadata": {"source": source, "chunk_index": chunk_index}
        })

    return chunks


chunks = split_into_chunks(cleaned_text)
dim_media = sum(len(c["text"]) for c in chunks) // len(chunks)
print(f"✅ Testo diviso in {len(chunks)} chunk")
print(f"   Dimensione media: {dim_media} caratteri")
print(f"   Primo chunk (anteprima): {chunks[0]['text'][:120]}...")

✅ Testo diviso in 18 chunk
   Dimensione media: 371 caratteri
   Primo chunk (anteprima): GUIDA DELLA LIBRERIA PAGINA & CO.
Via dei Libri 7, 10122 Torino (TO)
Aggiornamento: Settembre 2024 SEZIONE 1 - PROFILO D...


---
### STEP 3 — Indicizzazione su ChromaDB

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Indicizzazione su ChromaDB
# ─────────────────────────────────────────────────────────────────────────────
#
# TODO 1c — Crea un client ChromaDB in-memory, una funzione di embedding
#           e una collection, quindi indicizza tutti i chunk
#
# Istruzioni:
#   - Client: chromadb.Client()   ← in-memory, nessun file su disco
#   - Embedding: SentenceTransformerEmbeddingFunction con
#       model_name="paraphrase-multilingual-MiniLM-L12-v2"
#   - Collection: nome "libreria_kb", distanza coseno
#       metadata={"hnsw:space": "cosine"}
#   - Indicizza con collection.add(documents=[...], ids=[...], metadatas=[...])
#     dove ids="chunk_0", "chunk_1", ...
# ─────────────────────────────────────────────────────────────────────────────

# TODO: crea il client e la funzione di embedding
client       = chromadb.Client()  # chromadb.Client()
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)  # SentenceTransformerEmbeddingFunction(...)

# TODO: crea la collection
if any(c.name == "libreria_kb" for c in client.list_collections()):
    client.delete_collection("libreria_kb")

collection = client.create_collection(
    name="libreria_kb",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

# TODO: prepara le liste e indicizza i chunk
# documents = [...]
# ids       = [...]
# metadatas = [...]
# collection.add(...)

documents = [c['text'] for c in chunks]
ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [c['metadata'] for c in chunks]
collection.add(documents=documents, ids=ids, metadatas=metadatas)


# Verifica
if collection is None:
    print("⚠️  collection non creata")
else:
    print(f"✅ Collection '{collection.name}' creata — {collection.count()} chunk indicizzati")

✅ Collection 'libreria_kb' creata — 18 chunk indicizzati


---
### STEP 4 — Retrieval

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Retrieval
# ─────────────────────────────────────────────────────────────────────────────
#
# TODO 1d — Implementa la funzione retrieve()
#
# La funzione deve:
#   - Interrogare la collection con collection.query()
#   - Recuperare n_results chunk
#   - Restituire una lista di dict con chiavi:
#       "text"             → testo del chunk
#       "metadata"         → metadati del chunk
#       "similarity_score" → 1 - distanza (arrotondato a 4 decimali se vuoi)
# ─────────────────────────────────────────────────────────────────────────────

def retrieve(query: str, n_results: int = 4) -> list[dict]:
    """
    Recupera i chunk più rilevanti dalla knowledge base.

    Parametri
    ----------
    query     : domanda dell'utente
    n_results : numero di chunk da recuperare

    Ritorna
    -------
    lista di dict con 'text', 'metadata' e 'similarity_score'
    """
    # TODO: scrivi qui il corpo della funzione
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        retrieved.append({
            "text": doc,
            "metadata": meta,
            "similarity_score": round(1 - dist, 4)
        })

    return retrieved


# Verifica
risultati = retrieve("Quali sono gli orari di apertura della libreria?")
if risultati is None:
    print("⚠️  retrieve() restituisce None")
elif len(risultati) == 0:
    print("⚠️  Nessun chunk recuperato — controlla la query alla collection")
else:
    print(f"✅ Retrieval OK — {len(risultati)} chunk recuperati")
    for i, r in enumerate(risultati, 1):
        print(f"   [{i}] score={r['similarity_score']:.4f} | {r['text'][:80]}...")

✅ Retrieval OK — 4 chunk recuperati
   [1] score=0.5610 | Via dei Libri 7, 10122 Torino (TO)
Aggiornamento: Settembre 2024 SEZIONE 1 - PRO...
   [2] score=0.5421 | ro a bambino per sessione.
I laboratori sono condotti da Marta Ferretti, educatr...
   [3] score=0.5150 | rma dell'ordine. SEZIONE 5 - EVENTI E ATTIVITÀ Pagina & Co. organizza regolarmen...
   [4] score=0.5015 | gli di lettura
- Un segnalibro artigianale in omaggio all'iscrizione SEZIONE 4 -...


---
### STEP 5 — Generazione della risposta con Ollama

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Generazione della risposta
# ─────────────────────────────────────────────────────────────────────────────
#
# TODO 1e — Implementa la funzione generate_answer()
#
# La funzione deve:
#   1. Costruire il contesto unendo i testi dei chunk recuperati
#      (formato: "[Fonte 1]\n<testo>\n\n[Fonte 2]\n<testo>...")
#   2. Definire un system_message che:
#      - Presenta l'assistente come "Assistente della libreria Pagina & Co."
#      - Istruisce a rispondere SOLO con le informazioni nel contesto
#      - Se l'informazione non è presente, rispondere:
#        "Non ho questa informazione — ti consiglio di contattarci su info@paginaeco.it"
#   3. Chiamare ollama_client.chat() con model=OLLAMA_MODEL
#   4. Restituire response["message"]["content"]
# ─────────────────────────────────────────────────────────────────────────────

def generate_answer(query: str, retrieved_chunks: list[dict]) -> str:
    """
    Genera una risposta usando Ollama con il contesto dei chunk recuperati.

    Parametri
    ----------
    query            : domanda dell'utente
    retrieved_chunks : lista di dict con 'text' e 'metadata'

    Ritorna
    -------
    risposta generata dall'LLM
    """

    context = "\n\n".join(
        f"[Fonte {i+1}]\n{c['text']}" for i, c in enumerate(retrieved_chunks)
    )

    system_message = (
        "Sei l'assistente virtuale della libreria Pagina & Co. di Torino. "
        "Rispondi alle domande dei clienti basandoti ESCLUSIVAMENTE sulle "
        "informazioni fornite nel contesto. "
        "Se l'informazione non è presente nel contesto, rispondi: "
        "'Non ho questa informazione — ti consiglio di contattarci su info@paginaeco.it'. "
        "Rispondi sempre in italiano, in modo cordiale e professionale."
    )
    # TODO: scrivi qui il corpo della funzione
    user_message = f"""CONTESTO:

{context}

---
DOMANDA: {query}"""

    response = ollama_client.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        options={"temperature": 0}
    )
    return response["message"]["content"]


# ─────────────────────────────────────────────────────────────────────────────
# TODO 1f — Implementa la funzione libreria_chatbot() che unisce retrieve + generate
#
# La funzione deve:
#   1. Chiamare retrieve(query, n_results)
#   2. Stampare i chunk trovati con il rispettivo score
#   3. Chiamare generate_answer(query, retrieved)
#   4. Stampare e restituire la risposta
# ─────────────────────────────────────────────────────────────────────────────

def libreria_chatbot(query: str, n_results: int = 4) -> str:
    """
    Pipeline RAG completa: retrieve + generate.

    Parametri
    ----------
    query     : domanda del cliente
    n_results : chunk da recuperare

    Ritorna
    -------
    risposta dell'assistente della libreria
    """
    # TODO: scrivi qui il corpo della funzione
    retrieved = retrieve(query, n_results=n_results)
    print(f"\nChunk recuperati ({len(retrieved)}):")
    for i, r in enumerate(retrieved, 1):
        print(f"   [{i}] Score: {r['similarity_score']:.4f} | {r['text'][:100]}...")

    answer = generate_answer(query, retrieved)
    print(f"\nPagina & Co.:\n{answer}")
    return answer

---
### ✅ Test finale — Esercizio 1

Esegui la cella qui sotto per testare il tuo assistente con 3 domande di verifica.
Se l'implementazione è corretta, ogni risposta dovrà contenere informazioni coerenti
con la guida della libreria Pagina & Co.

In [9]:
# ── TEST — Non modificare questa cella ───────────────────────────────────────

domande_test = [
    "A che ora chiude la libreria il sabato?",
    "Quanto costa iscriversi al club del libro?",
    "La spedizione è gratuita? Da quale importo?",
]

print("=" * 64)
print("🧪 TEST ESERCIZIO 1 — Assistente Pagina & Co.")
print("=" * 64)

errori = 0
for i, domanda in enumerate(domande_test, 1):
    print(f"\n[Domanda {i}] {domanda}")
    try:
        risposta = libreria_chatbot(domanda)
        if risposta is None or risposta.strip() == "":
            print("❌ libreria_chatbot() ha restituito una risposta vuota")
            errori += 1
        else:
            print(f"✅ Risposta ricevuta ({len(risposta)} caratteri)")
    except Exception as e:
        print(f"❌ Errore: {e}")
        errori += 1

print("\n" + "=" * 64)
if errori == 0:
    print("🎉 Esercizio 1 completato con successo! 📚")
else:
    print(f"⚠️  {errori} domanda/e non ha prodotto risposta — controlla i TODO")
print("=" * 64)

🧪 TEST ESERCIZIO 1 — Assistente Pagina & Co.

[Domanda 1] A che ora chiude la libreria il sabato?

Chunk recuperati (4):
   [1] Score: 0.5171 | Via dei Libri 7, 10122 Torino (TO)
Aggiornamento: Settembre 2024 SEZIONE 1 - PROFILO DELLA LIBRERIA ...
   [2] Score: 0.4555 | ro a bambino per sessione.
I laboratori sono condotti da Marta Ferretti, educatrice e illustratrice....
   [3] Score: 0.4456 | rma dell'ordine. SEZIONE 5 - EVENTI E ATTIVITÀ Pagina & Co. organizza regolarmente eventi culturali:...
   [4] Score: 0.4136 | gli di lettura
- Un segnalibro artigianale in omaggio all'iscrizione SEZIONE 4 - ORDINI E SPEDIZIONI...



Pagina & Co.:
La libreria Pagina & Co. è aperta dal lunedì al sabato, dalle 9:30 alle 19:30.
✅ Risposta ricevuta (78 caratteri)

[Domanda 2] Quanto costa iscriversi al club del libro?

Chunk recuperati (4):
   [1] Score: 0.6119 | ti.
Gli autori presenti nel 2024 includono Donatella Di Pietrantonio, Paolo Cognetti e Viola Ardone....
   [2] Score: 0.5058 | rcati tramite il servizio di ricerca antiquaria, attivo su appuntamento. SEZIONE 3 - PREZZI E SCONTI...
   [3] Score: 0.4944 | i): sconto fino al 15%
- Remainders (libri con prezzo di copertina barrato): nessun limite di sconto...
   [4] Score: 0.4738 | ro a bambino per sessione.
I laboratori sono condotti da Marta Ferretti, educatrice e illustratrice....



Pagina & Co.:
Il costo dell'iscrizione al club del libro è di 20 euro, che include 10 incontri, tè e biscotti offerti dalla libreria.
✅ Risposta ricevuta (119 caratteri)

[Domanda 3] La spedizione è gratuita? Da quale importo?

Chunk recuperati (4):
   [1] Score: 0.5874 | : 3-5 giorni lavorativi
- Titoli fuori catalogo o esauriti: tempi variabili, preventivo su richiesta...
   [2] Score: 0.4900 | i): sconto fino al 15%
- Remainders (libri con prezzo di copertina barrato): nessun limite di sconto...
   [3] Score: 0.4522 | uito, i posti sono limitati a 40 persone (prenotazione consigliata). SEZIONE 6 - POLITICHE E SERVIZI...
   [4] Score: 0.3827 | bri con dediche o evidenziazioni.
Per i libri acquistati online, il reso è gratuito entro 14 giorni....



Pagina & Co.:
La spedizione è gratuita per ordini pari o superiori a 25 euro.
✅ Risposta ricevuta (63 caratteri)

🎉 Esercizio 1 completato con successo! 📚


---
## 🔬 Bonus — Confronto con modello quantizzato

Ora che il chatbot funziona con `llama3.2` (versione di default, fp16 o q4),
proviamo a rispondere alle **stesse domande** usando una versione più aggressivamente
quantizzata — ad esempio `llama3.2:3b-instruct-q2_K` — e confrontiamo le risposte.

### Obiettivo

Osservare come la **quantizzazione influisce sulla qualità delle risposte** in un
contesto RAG reale:
- Le risposte sono ancora accurate rispetto alla knowledge base?
- Il tono e la completezza cambiano?
- Il tempo di risposta migliora?

### Prima di procedere

Assicurati di avere il modello quantizzato disponibile su Ollama:

```bash
ollama pull llama3.2:3b-instruct-q2_K
```

> **Nota:** puoi sostituire `llama3.2:3b-instruct-q2_K` con qualsiasi altro
> modello quantizzato che hai già scaricato (es. `llama3.2:3b-instruct-q4_0`,
> `mistral:7b-instruct-q4_K_M`, ecc.).

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# TODO — Confronto con modello quantizzato
#
# Istruzioni:
#   1. Definisci OLLAMA_MODEL_Q con il nome del modello quantizzato da testare
#   2. Implementa chatbot_quantizzato(), una funzione identica a libreria_chatbot()
#      ma che usa OLLAMA_MODEL_Q al posto di OLLAMA_MODEL
#   3. Esegui le stesse 3 domande di test su entrambi i modelli
#   4. Stampa le risposte affiancate per ogni domanda, con il tempo di risposta
# ─────────────────────────────────────────────────────────────────────────────

# TODO 1g — Scegli il modello quantizzato da testare
OLLAMA_MODEL_Q = "llama3.2:3b-instruct-q2_K"  # <-- cambia se necessario


# TODO 1h — Implementa chatbot_quantizzato()
#
# È identica a libreria_chatbot(), con un'unica differenza:
# la chiamata a generate_answer() deve usare OLLAMA_MODEL_Q.
#
# Suggerimento: puoi copiare il corpo di libreria_chatbot() e
# aggiungere il parametro modello a generate_answer(), oppure
# aggiungere un parametro opzionale `modello` a generate_answer()
# con default = OLLAMA_MODEL.

def chatbot_quantizzato(query: str, n_results: int = 4) -> str:
    """
    Pipeline RAG identica a libreria_chatbot(),
    ma usa il modello quantizzato OLLAMA_MODEL_Q.

    Parametri
    ----------
    query     : domanda del cliente
    n_results : chunk da recuperare

    Ritorna
    -------
    risposta generata dal modello quantizzato
    """
    # TODO: scrivi qui il corpo della funzione
    retrieved_chunks = retrieve(query, n_results=n_results)
    print(f"\nChunk recuperati ({len(retrieved_chunks)}):")
    for i, r in enumerate(retrieved_chunks, 1):
        print(f"   [{i}] Score: {r['similarity_score']:.4f} | {r['text'][:100]}...")

    context = "\n\n".join(
        f"[Fonte {i+1}]\n{c['text']}" for i, c in enumerate(retrieved_chunks)
    )

    system_message = (
        "Sei l'assistente virtuale della libreria Pagina & Co. di Torino. "
        "Rispondi alle domande dei clienti basandoti ESCLUSIVAMENTE sulle "
        "informazioni fornite nel contesto. "
        "Se l'informazione non è presente nel contesto, rispondi: "
        "'Non ho questa informazione — ti consiglio di contattarci su info@paginaeco.it'. "
        "Rispondi sempre in italiano, in modo cordiale e professionale."
    )

    user_message = f"""CONTESTO:

{context}

---
DOMANDA: {query}"""

    response = ollama_client.chat(
        model=OLLAMA_MODEL_Q,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        options={"temperature": 0}
    )
    answer = response["message"]["content"]
    print(f"\nPagina & Co. (quantizzato):\n{answer}")
    return answer

In [11]:
# ── TEST CONFRONTO — Non modificare questa cella ──────────────────────────────

domande_test = [
    "A che ora chiude la libreria il sabato?",
    "Quanto costa iscriversi al club del libro?",
    "La spedizione è gratuita? Da quale importo?",
]

print("=" * 72)
print(f"  CONFRONTO RISPOSTE: {OLLAMA_MODEL}  vs  {OLLAMA_MODEL_Q}")
print("=" * 72)

for i, domanda in enumerate(domande_test, 1):
    print(f"\n{'─'*72}")
    print(f"❓ Domanda {i}: {domanda}")
    print(f"{'─'*72}")

    # Modello base
    t0 = time.time()
    risposta_base = libreria_chatbot(domanda)
    t_base = time.time() - t0

    # Modello quantizzato
    t0 = time.time()
    risposta_q = chatbot_quantizzato(domanda)
    t_q = time.time() - t0

    # Stampa affiancata
    sep = "─" * 34
    print(f"\n🟦 [{OLLAMA_MODEL}]  ({t_base:.1f}s)")
    print(sep)
    if risposta_base:
        print(textwrap.fill(risposta_base.strip(), width=68))
    else:
        print("  ⚠️  Nessuna risposta")

    print(f"\n🟨 [{OLLAMA_MODEL_Q}]  ({t_q:.1f}s)")
    print(sep)
    if risposta_q:
        print(textwrap.fill(risposta_q.strip(), width=68))
    else:
        print("  ⚠️  Nessuna risposta")

  CONFRONTO RISPOSTE: llama3.2  vs  llama3.2:3b-instruct-q2_K

────────────────────────────────────────────────────────────────────────
❓ Domanda 1: A che ora chiude la libreria il sabato?
────────────────────────────────────────────────────────────────────────

Chunk recuperati (4):
   [1] Score: 0.5171 | Via dei Libri 7, 10122 Torino (TO)
Aggiornamento: Settembre 2024 SEZIONE 1 - PROFILO DELLA LIBRERIA ...
   [2] Score: 0.4555 | ro a bambino per sessione.
I laboratori sono condotti da Marta Ferretti, educatrice e illustratrice....
   [3] Score: 0.4456 | rma dell'ordine. SEZIONE 5 - EVENTI E ATTIVITÀ Pagina & Co. organizza regolarmente eventi culturali:...
   [4] Score: 0.4136 | gli di lettura
- Un segnalibro artigianale in omaggio all'iscrizione SEZIONE 4 - ORDINI E SPEDIZIONI...



Pagina & Co.:
La libreria Pagina & Co. è aperta dal lunedì al sabato, dalle 9:30 alle 19:30.

Chunk recuperati (4):
   [1] Score: 0.5171 | Via dei Libri 7, 10122 Torino (TO)
Aggiornamento: Settembre 2024 SEZIONE 1 - PROFILO DELLA LIBRERIA ...
   [2] Score: 0.4555 | ro a bambino per sessione.
I laboratori sono condotti da Marta Ferretti, educatrice e illustratrice....
   [3] Score: 0.4456 | rma dell'ordine. SEZIONE 5 - EVENTI E ATTIVITÀ Pagina & Co. organizza regolarmente eventi culturali:...
   [4] Score: 0.4136 | gli di lettura
- Un segnalibro artigianale in omaggio all'iscrizione SEZIONE 4 - ORDINI E SPEDIZIONI...



Pagina & Co. (quantizzato):
Non ho questa informazione — ti consiglio di contattarci su info@paginaeco.it.

🟦 [llama3.2]  (0.9s)
──────────────────────────────────
La libreria Pagina & Co. è aperta dal lunedì al sabato, dalle 9:30
alle 19:30.

🟨 [llama3.2:3b-instruct-q2_K]  (5.7s)
──────────────────────────────────
Non ho questa informazione — ti consiglio di contattarci su
info@paginaeco.it.

────────────────────────────────────────────────────────────────────────
❓ Domanda 2: Quanto costa iscriversi al club del libro?
────────────────────────────────────────────────────────────────────────

Chunk recuperati (4):
   [1] Score: 0.6119 | ti.
Gli autori presenti nel 2024 includono Donatella Di Pietrantonio, Paolo Cognetti e Viola Ardone....
   [2] Score: 0.5058 | rcati tramite il servizio di ricerca antiquaria, attivo su appuntamento. SEZIONE 3 - PREZZI E SCONTI...
   [3] Score: 0.4944 | i): sconto fino al 15%
- Remainders (libri con prezzo di copertina barrato): nessun limite di sconto


Pagina & Co.:
Il costo dell'iscrizione al club del libro è di 20 euro, che include 10 incontri, tè e biscotti offerti dalla libreria.

Chunk recuperati (4):
   [1] Score: 0.6119 | ti.
Gli autori presenti nel 2024 includono Donatella Di Pietrantonio, Paolo Cognetti e Viola Ardone....
   [2] Score: 0.5058 | rcati tramite il servizio di ricerca antiquaria, attivo su appuntamento. SEZIONE 3 - PREZZI E SCONTI...
   [3] Score: 0.4944 | i): sconto fino al 15%
- Remainders (libri con prezzo di copertina barrato): nessun limite di sconto...
   [4] Score: 0.4738 | ro a bambino per sessione.
I laboratori sono condotti da Marta Ferretti, educatrice e illustratrice....



Pagina & Co. (quantizzato):
Rispondendo al contesto, posso rispondere in base alle informazioni fornite. 

Per il contesto 1, non ho informazioni sul prezzo dell'iscrizione al club. Ti consiglio di contattarci su info@paginaeco.it per ulteriori informazioni.

🟦 [llama3.2]  (7.4s)
──────────────────────────────────
Il costo dell'iscrizione al club del libro è di 20 euro, che include
10 incontri, tè e biscotti offerti dalla libreria.

🟨 [llama3.2:3b-instruct-q2_K]  (6.7s)
──────────────────────────────────
Rispondendo al contesto, posso rispondere in base alle informazioni
fornite.   Per il contesto 1, non ho informazioni sul prezzo
dell'iscrizione al club. Ti consiglio di contattarci su
info@paginaeco.it per ulteriori informazioni.

────────────────────────────────────────────────────────────────────────
❓ Domanda 3: La spedizione è gratuita? Da quale importo?
────────────────────────────────────────────────────────────────────────

Chunk recuperati (4):
   [1] Score: 0.5874 | : 3-5 gi


Pagina & Co.:
La spedizione è gratuita per ordini pari o superiori a 25 euro.

Chunk recuperati (4):
   [1] Score: 0.5874 | : 3-5 giorni lavorativi
- Titoli fuori catalogo o esauriti: tempi variabili, preventivo su richiesta...
   [2] Score: 0.4900 | i): sconto fino al 15%
- Remainders (libri con prezzo di copertina barrato): nessun limite di sconto...
   [3] Score: 0.4522 | uito, i posti sono limitati a 40 persone (prenotazione consigliata). SEZIONE 6 - POLITICHE E SERVIZI...
   [4] Score: 0.3827 | bri con dediche o evidenziazioni.
Per i libri acquistati online, il reso è gratuito entro 14 giorni....



Pagina & Co. (quantizzato):
Non ho questa informazione — ti consiglio di contattarci su info@paginaeco.it.

🟦 [llama3.2]  (6.9s)
──────────────────────────────────
La spedizione è gratuita per ordini pari o superiori a 25 euro.

🟨 [llama3.2:3b-instruct-q2_K]  (5.4s)
──────────────────────────────────
Non ho questa informazione — ti consiglio di contattarci su
info@paginaeco.it.


---
# 📝 Esercizio 2 — Agente di analisi con Tool Calling
---

## Descrizione

Devi costruire un **agente AI** capace di rispondere a domande che richiedono
l'uso di strumenti (tool). L'agente ha a disposizione **3 tool**:

| Tool | Descrizione | Input |
|------|-------------|-------|
| `calcola_errore_quantizzazione` | Calcola l'errore medio assoluto tra un valore float e la sua versione quantizzata a N bit | `valore: float`, `bits: int` |
| `confronta_modelli` | Restituisce dati fittizi di benchmark (velocità e qualità) di modelli Ollama a diverse quantizzazioni | `modello: str` |
| `salva_report` | Salva un testo in un file `.txt` e restituisce il percorso | `contenuto: str`, `nome_file: str` |

Il ciclo dell'agente segue il pattern **ReAct** visto in `03_introduzione_ai_agenti.ipynb`:

```
 ┌──────────────────────────────────────────────┐
 │             REASONING LOOP                   │
 │                                              │
 │  Input → LLM ──► tool_call? ──► esegui tool  │
 │                  │                    │      │
 │                  │ no          ToolMessage   │
 │                  ▼                    │      │
 │             Risposta finale ◄──────────┘      │
 └──────────────────────────────────────────────┘
```

---

## Cosa devi fare

1. **Definire i 3 tool** usando il decorator `@tool` di LangChain
2. **Bindare i tool all'LLM** con `llm.bind_tools([...])`
3. **Implementare il reasoning loop** (ciclo agente)
4. **Testare l'agente** con i task forniti

---
### Setup LLM — Esercizio 2

In [12]:
OLLAMA_MODEL = "qwen2.5"

In [13]:
# ── LLM per l'agente (temperature=0 per comportamento deterministico) ─────────
llm_agente = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=BASE_URL,
    temperature=0,
)

try:
    test = llm_agente.invoke("Rispondi solo: Pronto!")
    print(f"✅ LLM agente pronto: {test.content.strip()}")
except Exception as e:
    print(f"❌ Errore: {e}")

✅ LLM agente pronto: Pronto!


---
### Definizione dei Tool

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# TODO 2a — Definisci il tool `calcola_errore_quantizzazione`
#
# Il tool deve:
#   1. Ricevere un valore float e un numero di bit (default 8)
#   2. Calcolare il range di quantizzazione: L = 2^bits livelli
#   3. Calcolare delta = (abs(valore) * 2) / (L - 1)
#      (approssima il range come [-|x|, +|x|] per semplicità)
#   4. Calcolare l'errore massimo teorico = delta / 2
#   5. Restituire una stringa descrittiva con:
#      - il valore originale
#      - il numero di bit
#      - il numero di livelli L
#      - delta
#      - l'errore massimo atteso
#
# Usa il decorator @tool con una docstring descrittiva in inglese
# (LangChain usa la docstring come descrizione del tool per l'LLM)
# ─────────────────────────────────────────────────────────────────────────────

# TODO: scrivi qui il tool calcola_errore_quantizzazione

@tool
def calcola_errore_quantizzazione(valore: float, bits: int = 8) -> str:
    """Calculate the quantization step and maximum theoretical error for a value and bit count."""
    L = 2 ** bits
    delta = (abs(valore) * 2) / (L - 1)
    errore_massimo = delta / 2
    return (
        f"Valore originale: {valore}; bit: {bits}; livelli L: {L}; "
        f"delta: {delta:.6f}; errore massimo teorico: {errore_massimo:.6f}."
    )


# ─────────────────────────────────────────────────────────────────────────────
# TODO 2b — Definisci il tool `confronta_modelli`
#
# Il tool deve restituire dati fittizi di benchmark per un dato modello Ollama.
# Usa questo dizionario come fonte dati:
#
BENCHMARK_DATA = {
    "llama3.2": {
        "q2_K":  {"velocita_token_s": 65,  "qualita_score": 6.1, "memoria_GB": 1.2},
        "q4_K_M":{"velocita_token_s": 48,  "qualita_score": 7.8, "memoria_GB": 2.0},
        "q8_0":  {"velocita_token_s": 28,  "qualita_score": 8.9, "memoria_GB": 3.8},
        "f16":   {"velocita_token_s": 12,  "qualita_score": 9.5, "memoria_GB": 6.4},
    },
    "mistral": {
        "q2_K":  {"velocita_token_s": 55,  "qualita_score": 5.9, "memoria_GB": 2.8},
        "q4_K_M":{"velocita_token_s": 38,  "qualita_score": 7.5, "memoria_GB": 4.1},
        "q8_0":  {"velocita_token_s": 18,  "qualita_score": 8.7, "memoria_GB": 7.7},
        "f16":   {"velocita_token_s": 8,   "qualita_score": 9.3, "memoria_GB": 14.5},
    },
}
#
# Il tool deve:
#   - Ricevere il nome del modello ("llama3.2" o "mistral")
#   - Se il modello non è in BENCHMARK_DATA, restituire un messaggio di errore
#   - Altrimenti, restituire una stringa formattata con tutti i dati disponibili
#     per ogni livello di quantizzazione (velocità token/s, qualità score, memoria GB)
# ─────────────────────────────────────────────────────────────────────────────

# TODO: scrivi qui il tool confronta_modelli
@tool
def confronta_modelli(modello: str) -> str:
    """
    Restituisce i dati di riferimento (velocità, punteggio di qualità, memoria) per un determinato modello Ollama a diversi livelli di quantizzazione.

    Args:
        modello: nome del modello, “llama3.2” o “mistral”

    Returns:
        Una stringa formattata contenente i dati di riferimento per ciascun livello di quantizzazione oppure un messaggio di errore se il modello non viene trovato.
    """
    if modello not in BENCHMARK_DATA:
        modelli_disponibili = ", ".join(BENCHMARK_DATA.keys())
        return (
            f"Modello '{modello}' non trovato nel database. "
            f"Modelli disponibili: {modelli_disponibili}"
        )

    righe = [f"Benchmark per il modello: {modello}\n"]
    righe.append(f"{'Quantizzazione':<12} {'Velocità (tok/s)':>17} {'Qualità (0-10)':>15} {'Memoria (GB)':>13}")
    righe.append("─" * 60)

    for quant, dati in BENCHMARK_DATA[modello].items():
        righe.append(
            f"{quant:<12} {dati['velocita_token_s']:>17}  "
            f"{dati['qualita_score']:>14.1f}  "
            f"{dati['memoria_GB']:>12.1f}"
        )

    return "\n".join(righe)

# ─────────────────────────────────────────────────────────────────────────────
# TODO 2c — Definisci il tool `salva_report`
#
# Il tool deve:
#   - Ricevere una stringa `contenuto` e un `nome_file` (default "report.txt")
#   - Assicurarsi che il nome file termini con ".txt"
#   - Scrivere il contenuto nel file con encoding utf-8
#   - Restituire una stringa che conferma il salvataggio e indica il percorso
# ─────────────────────────────────────────────────────────────────────────────

# TODO: scrivi qui il tool salva_report
import os

@tool
def salva_report(contenuto: str, nome_file: str = "report.txt") -> str:
    """Save the provided report to a UTF-8 text file and return its path."""
    if not nome_file.endswith(".txt"):
        nome_file += ".txt"
    with open(nome_file, "w", encoding="utf-8") as f:
        f.write(contenuto)
    return f"Report salvato in: {os.path.abspath(nome_file)}"


---
### Implementazione del Reasoning Loop

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# TODO 2d — Binda i tool all'LLM e crea la mappa dei tool
#
# Istruzioni:
#   1. Crea la lista tools = [...] con i 3 tool definiti sopra
#   2. Binda i tool: llm_con_tools = llm_agente.bind_tools(tools)
#   3. Crea un dizionario tool_map che mappa nome_tool → funzione_tool
#      Suggerimento: ogni @tool ha un attributo .name
# ─────────────────────────────────────────────────────────────────────────────

# TODO: definisci tools, llm_con_tools e tool_map
tools        = [calcola_errore_quantizzazione, confronta_modelli, salva_report]    # <-- lista dei tool
llm_con_tools = llm_agente.bind_tools(tools) # <-- llm_agente.bind_tools(tools)
tool_map     = {t.name: t for t in tools}    # <-- {tool.name: tool for tool in tools}

print(f"Tool registrati: {[t.name for t in tools]}")

Tool registrati: ['calcola_errore_quantizzazione', 'confronta_modelli', 'salva_report']


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# TODO 2e — Implementa il reasoning loop dell'agente
#
# La funzione esegui_agente() deve:
#   1. Inizializzare la lista messaggi con un SystemMessage + HumanMessage
#      Il system message deve dire all'agente:
#      - Chi è (esperto di quantizzazione LLM)
#      - Che ha a disposizione dei tool da usare per rispondere
#      - Di essere preciso e citare i dati numerici nella risposta finale
#
#   2. Eseguire il loop (max_steps iterazioni):
#      a. Invocare llm_con_tools con la lista messaggi corrente
#      b. Aggiungere la risposta dell'LLM alla lista messaggi
#      c. Se la risposta contiene tool_calls:
#         - Per ogni tool_call:
#           * Recuperare il tool da tool_map usando tool_call.name  (o ["name"])
#           * Invocare il tool con tool_call.args  (o ["args"])
#           * Creare un ToolMessage con il risultato e il tool_call_id corretto
#           * Aggiungerlo alla lista messaggi
#         - Stampare quali tool sono stati invocati
#      d. Se non ci sono tool_calls: l'agente ha finito → break
#
#   3. Restituire il contenuto dell'ultimo messaggio AIMessage
#
# Suggerimento: guarda come è strutturato il loop in 03_introduzione_ai_agenti.ipynb
# ─────────────────────────────────────────────────────────────────────────────

def esegui_agente(task: str, max_steps: int = 6) -> str:
    """
    Esegue il reasoning loop dell'agente per completare un task.

    Parametri
    ----------
    task      : descrizione del task da completare
    max_steps : numero massimo di iterazioni del loop

    Ritorna
    -------
    risposta finale dell'agente
    """
    print(f"\n{'='*64}")
    print(f"🤖 Task: {task}")
    print(f"{'='*64}")

    # TODO: scrivi qui il reasoning loop
    messaggi = [
        SystemMessage(content=(
            "Sei un esperto di quantizzazione LLM. "
            "Hai a disposizione tool per fare calcoli, confrontare modelli e salvare report. "
            "Usali per rispondere alle richieste. "
            "Sii preciso e cita i dati numerici dei tool nella risposta finale."
            " Nei calcoli, delta è il passo tra due livelli consecutivi di quantizzazione. "
            "Non è l'errore effettivo del singolo valore: delta/2 è il limite massimo teorico "
            "dell'errore con arrotondamento al livello più vicino."
            " Per creare un report, prima usa confronta_modelli per ciascun modello richiesto. "
            "Attendi i risultati nei ToolMessage prima di scrivere il report. "
            "Chiama salva_report soltanto in un turno successivo, con un testo che includa "
            "per ogni modello e livello di quantizzazione la velocità in token/s, "
            "il punteggio di qualità e la memoria in GB. "
            "Il contenuto deve includere tutti questi dati, oltre al titolo. "
            "Conferma il salvataggio soltanto dopo avere ricevuto il risultato di salva_report."
        )),
        HumanMessage(content=task)
    ]

    for step in range(max_steps):
        risposta = llm_con_tools.invoke(messaggi)
        messaggi.append(risposta)

        if not risposta.tool_calls:
            return risposta.content

        for tool_call in risposta.tool_calls:
            nome_tool = tool_call["name"]
            argomenti = tool_call["args"]
            print(f"Tool invocato: {nome_tool} | Argomenti: {argomenti}")
            risultato_tool = tool_map[nome_tool].invoke(argomenti)
            messaggi.append(ToolMessage(
                content=str(risultato_tool),
                tool_call_id=tool_call["id"]
            ))

    return "Limite massimo di iterazioni raggiunto senza risposta finale."

---
### ✅ Test finale — Esercizio 2

Esegui la cella qui sotto per testare il tuo agente con 3 task progressivi.
Ogni task richiede l'uso di uno o più tool.

In [17]:
# ── TEST — Non modificare questa cella ───────────────────────────────────────

task_test = [
    # Task 1: usa calcola_errore_quantizzazione
    "Calcola l'errore di quantizzazione per il valore 3.14159 a 4 bit e a 8 bit. "
    "Poi spiegami la differenza.",

    # Task 2: usa confronta_modelli
    "Confronta le prestazioni del modello llama3.2 a diversi livelli di quantizzazione. "
    "Quale offre il miglior compromesso tra velocità e qualità?",

    # Task 3: usa confronta_modelli + salva_report
    "Analizza i benchmark di llama3.2 e mistral, poi salva un report di confronto "
    "nel file 'confronto_modelli.txt'.",
]

print("=" * 64)
print("🧪 TEST ESERCIZIO 2 — Agente di analisi")
print("=" * 64)

errori = 0
for i, task in enumerate(task_test, 1):
    print(f"\n[Task {i}]")
    try:
        risposta = esegui_agente(task)
        if risposta is None or (isinstance(risposta, str) and risposta.strip() == ""):
            print("❌ esegui_agente() ha restituito una risposta vuota")
            errori += 1
        else:
            print(f"\n💬 Risposta finale:")
            print(textwrap.fill(str(risposta).strip(), width=70))
            print(f"\n✅ Task {i} completato")
    except Exception as e:
        print(f"❌ Errore durante task {i}: {e}")
        errori += 1

print("\n" + "=" * 64)
if errori == 0:
    print("🎉 Esercizio 2 completato con successo!")
else:
    print(f"⚠️  {errori} task non completato/i — controlla i TODO")
print("=" * 64)

🧪 TEST ESERCIZIO 2 — Agente di analisi

[Task 1]

🤖 Task: Calcola l'errore di quantizzazione per il valore 3.14159 a 4 bit e a 8 bit. Poi spiegami la differenza.


Tool invocato: calcola_errore_quantizzazione | Argomenti: {'valore': 3.14159, 'bits': 4}
Tool invocato: calcola_errore_quantizzazione | Argomenti: {'valore': 3.14159, 'bits': 8}



💬 Risposta finale:
Per il valore 3.14159 con 4 bit di quantizzazione, il delta (passo tra
due livelli consecutivi) è di circa 0.418879, e il limite massimo
teorico dell'errore è di circa 0.209439.   Invece, con 8 bit di
quantizzazione, il delta diminuisce a circa 0.024640, e il limite
massimo teorico dell'errore si riduce a circa 0.012320.  In sintesi,
aumentare il numero di bit di quantizzazione riduce il delta e, di
conseguenza, l'errore massimo teorico. Questo significa che con più
bit, la quantizzazione diventa più precisa, ma allo stesso tempo
richiede più memoria e potrebbe influire sulla velocità del calcolo.

✅ Task 1 completato

[Task 2]

🤖 Task: Confronta le prestazioni del modello llama3.2 a diversi livelli di quantizzazione. Quale offre il miglior compromesso tra velocità e qualità?


Tool invocato: confronta_modelli | Argomenti: {'modello': 'llama3.2'}


Tool invocato: salva_report | Argomenti: {'nome_file': 'report_llama3.2.txt', 'contenuto': 'Report per il modello llama3.2\n\nQuantizzazione  Velocità (tok/s)  Qualità (0-10)  Memoria (GB)\n────────────────────────────────────────────────────────────\nq2_K                        65             6.1           1.2\nq4_K_M                      48             7.8           2.0\nq8_0                        28             8.9           3.8\nf16                         12             9.5           6.4\n\nIl livello di quantizzazione q4_K_M offre il miglior compromesso tra velocità e qualità, con una velocità di 48 token/s e un punteggio di qualità di 7.8, utilizzando 2.0 GB di memoria.'}



💬 Risposta finale:
Il report sulle prestazioni del modello llama3.2 a diversi livelli di
quantizzazione è stato salvato con successo. Ecco i dettagli inclusi
nel report:  **Report per il modello llama3.2**  | Quantizzazione |
Velocità (tok/s) | Punteggio di qualità (0-10) | Memoria (GB) | |-----
-----------|-----------------|----------------------------|-----------
--| | q2_K           | 65              | 6.1                        |
1.2         | | q4_K_M         | 48              | 7.8
| 2.0         | | q8_0           | 28              | 8.9
| 3.8         | | f16            | 12              | 9.5
| 6.4         |  Il livello di quantizzazione **q4_K_M** offre il
miglior compromesso tra velocità e qualità, con una velocità di 48
token/s e un punteggio di qualità di 7.8, utilizzando 2.0 GB di
memoria.  Il report è stato salvato in:
C:\Users\A1621apulia\Desktop\GENERATIVE AI - PRIMO ANNO\CONSEGNA\PARTE
3\report_llama3.2.txt.

✅ Task 2 completato

[Task 3]

🤖 Task: Analizza i benchmark 

Tool invocato: confronta_modelli | Argomenti: {'modello': 'llama3.2'}
Tool invocato: confronta_modelli | Argomenti: {'modello': 'mistral'}


Tool invocato: salva_report | Argomenti: {'contenuto': 'Confronto dei modelli:\n\n**Llama3.2**\n\nQuantizzazione | Velocità (tok/s) | Qualità (0-10) | Memoria (GB)\n----------------|----------------|---------------|-------------\nq2_K            | 65             | 6.1           | 1.2\nq4_K_M          | 48             | 7.8           | 2.0\nq8_0            | 28             | 8.9           | 3.8\nf16             | 12             | 9.5           | 6.4\n\n**Mistral**\n\nQuantizzazione | Velocità (tok/s) | Qualità (0-10) | Memoria (GB)\n----------------|----------------|---------------|-------------\nq2_K            | 55             | 5.9           | 2.8\nq4_K_M          | 38             | 7.5           | 4.1\nq8_0            | 18             | 8.7           | 7.7\nf16             | 8              | 9.3           | 14.5', 'nome_file': 'confronto_modelli.txt'}



💬 Risposta finale:
Il report di confronto tra i modelli Llama3.2 e Mistral è stato
salvato nel file `confronto_modelli.txt`. Ecco i dettagli:
**Llama3.2**  | Quantizzazione | Velocità (tok/s) | Qualità (0-10) |
Memoria (GB) |
|----------------|-----------------|---------------|-------------| |
q2_K           | 65              | 6.1           | 1.2         | |
q4_K_M         | 48              | 7.8           | 2.0         | |
q8_0           | 28              | 8.9           | 3.8         | | f16
| 12              | 9.5           | 6.4         |  **Mistral**  |
Quantizzazione | Velocità (tok/s) | Qualità (0-10) | Memoria (GB) |
|----------------|-----------------|---------------|-------------| |
q2_K           | 55              | 5.9           | 2.8         | |
q4_K_M         | 38              | 7.5           | 4.1         | |
q8_0           | 18              | 8.7           | 7.7         | | f16
| 8               | 9.3           | 14.5        |  Il file
`confronto_modelli.txt` contiene